# Projet - Prédiction Sinistre

![](https://imgs.search.brave.com/WXdC7EEYkRxHJG8uX1e-SrHqaRfVUHiMHm_aHKrdVjk/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly9tZWRp/YXMubGVjb21wYXJh/dGV1cmFzc3VyYW5j/ZS5jb20vTENBX0Nv/bnRlbnRzL29sZC9h/c3N1cmFuY2UtdGF4/aS1xdWUtZmFpcmUt/Y2FzLXNpbmlzdHJl/LWRlZ3JhZGF0aW9u/LXZlaGljdWxlLndl/YnA)

## Description et objectif du projet :
L'objectif de ce projet est d'analyser l'ensemble des données de réclamations d'assurance afin de développer un modèle prédictif capable de détecter les réclamations potentiellement frauduleuses. 

Dans l'identification des indicateurs clés associés à la fraude, ce modèle vise à aider les compagnies d'assurance à réduire les pertes financières et à améliorer l'efficacité de leurs processus de validation des réclamations.

C'est donc un projet d'*`apprentissage supervisé`* et il traite une problématique de *`classification binaire`* (dite à 2 classes ou 2 variables)
> Le but est d'avoir le *`score F1`* le plus grand possible.

## 1. Chargement et exploration de données

In [2]:
# Importation des packages nécessaires
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import plotly.express as px
from wordcloud import WordCloud

In [3]:
# Lecture des données
data = pd.read_csv("../data/raw/insurance_claims.csv")

# Affichage des 5 premières lignes du DataFrame
data.head()

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,_c39
0,328,48,521585,2014-10-17,OH,250/500,1000,1406.91,0,466132,...,YES,71610,6510,13020,52080,Saab,92x,2004,Y,NaN
1,228,42,342868,2006-06-27,IN,250/500,2000,1197.22,5000000,468176,...,?,5070,780,780,3510,Mercedes,E400,2007,Y,NaN
2,134,29,687698,2000-09-06,OH,100/300,2000,1413.14,5000000,430632,...,NO,34650,7700,3850,23100,Dodge,RAM,2007,N,NaN
3,256,41,227811,1990-05-25,IL,250/500,2000,1415.74,6000000,608117,...,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y,NaN
4,228,44,367455,2014-06-06,IL,500/1000,1000,1583.91,6000000,610706,...,NO,6500,1300,650,4550,Accura,RSX,2009,N,NaN


In [4]:
# Taille du DataFrame
print(f"Ce DataFrame contient {data.shape[0]} lignes et {data.shape[1]} colonnes.")

Ce DataFrame contient 1000 lignes et 40 colonnes.


## 2. Analyse Fondamentale de Données

Il s'agira dans cette partie de comprendre la signification des variables, et surtout le contexte qui entoure ces données.
Cette étape est complémentaire à l'*`Analyse Technique`* qui suit.

In [7]:
# Information sur les données
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   months_as_customer           1000 non-null   int64  
 1   age                          1000 non-null   int64  
 2   policy_number                1000 non-null   int64  
 3   policy_bind_date             1000 non-null   str    
 4   policy_state                 1000 non-null   str    
 5   policy_csl                   1000 non-null   str    
 6   policy_deductable            1000 non-null   int64  
 7   policy_annual_premium        1000 non-null   float64
 8   umbrella_limit               1000 non-null   int64  
 9   insured_zip                  1000 non-null   int64  
 10  insured_sex                  1000 non-null   str    
 11  insured_education_level      1000 non-null   str    
 12  insured_occupation           1000 non-null   str    
 13  insured_hobbies              1

In [8]:
# Vérification des valeurs manquantes
print(f"Nous avons : {data.isna().sum().sum()} valeurs manquantes.")

print("\nCes valeurs manquantes sont réparties comme suit :")

# Parcours des colonnes pour afficher les valeurs manquantes
for col in data.columns:
    missing_count = data[col].isna().sum()
    if missing_count > 0:
        # Affichage du nombre de valeurs manquantes pour chaque colonne concernée
        print(f"{col} : {missing_count} valeurs manquantes")

Nous avons : 1091 valeurs manquantes.

Ces valeurs manquantes sont réparties comme suit :
authorities_contacted : 91 valeurs manquantes
_c39 : 1000 valeurs manquantes


> *On remarque que la colonne `_c39` est vide (ne contient acune valeur), elle pourra être supprimée et `authorities_contact` contient 91 valeurs manquantes sur 1000, il reste alors 909 valeurs qui sont assez suffisantes pour l'analyse.*

In [9]:
# Suppression de la colonne '_c39' car elle ne contient aucune valeur
data = data.drop(columns = "_c39")

In [10]:
# Vérification de la suppression de _c39
print(data.columns)

Index(['months_as_customer', 'age', 'policy_number', 'policy_bind_date',
       'policy_state', 'policy_csl', 'policy_deductable',
       'policy_annual_premium', 'umbrella_limit', 'insured_zip', 'insured_sex',
       'insured_education_level', 'insured_occupation', 'insured_hobbies',
       'insured_relationship', 'capital-gains', 'capital-loss',
       'incident_date', 'incident_type', 'collision_type', 'incident_severity',
       'authorities_contacted', 'incident_state', 'incident_city',
       'incident_location', 'incident_hour_of_the_day',
       'number_of_vehicles_involved', 'property_damage', 'bodily_injuries',
       'witnesses', 'police_report_available', 'total_claim_amount',
       'injury_claim', 'property_claim', 'vehicle_claim', 'auto_make',
       'auto_model', 'auto_year', 'fraud_reported'],
      dtype='str')


In [11]:
# Vérification des valeurs en double dans le DataFrame
print(f"Nombre de lignes en double dans le DataFrame : {data.duplicated().sum()}")

Nombre de lignes en double dans le DataFrame : 0


In [12]:
# Liste des colonnes contenant des dates étant au format "string"
date_columns = ["incident_date", "policy_bind_date",]

# Parcourir les colonnes de date pour les convertir en datetime
for col in date_columns:
    data[col] = pd.to_datetime(data[col], errors="coerce") # Conversion

In [13]:
# Vérification de l'application de la conversion
print(data[date_columns].dtypes)

incident_date       datetime64[us]
policy_bind_date    datetime64[us]
dtype: object


> *La conversion des colonnes ayant des dates permettra une bonne analyse, notamment l'analyse temporelle envue de découvrir les tendances dans les incidents.*

In [14]:
# Sélection des variables numériques et stats descriptives avec la sortie en transposée
num_cols = data.select_dtypes(include=np.number).describe().T

# Affichage des stats par ordre croissant suivant la moyenne
num_cols.sort_values("mean", ascending = True)

,count,mean,std,min,25%,50%,75%,max
capital-loss,1000.0,-2.679370e+04,2.810410e+04,-111100.00,-51500.0000,-23250.0,0.000,0.00
bodily_injuries,1000.0,9.920000e-01,8.201272e-01,0.00,0.0000,1.0,2.000,2.00
witnesses,1000.0,1.487000e+00,1.111335e+00,0.00,1.0000,1.0,2.000,3.00
number_of_vehicles_involved,1000.0,1.839000e+00,1.018880e+00,1.00,1.0000,1.0,3.000,4.00
incident_hour_of_the_day,1000.0,1.164400e+01,6.951373e+00,0.00,6.0000,12.0,17.000,23.00
age,1000.0,3.894800e+01,9.140287e+00,19.00,32.0000,38.0,44.000,64.00
months_as_customer,1000.0,2.039540e+02,1.151132e+02,0.00,115.7500,199.5,276.250,479.00
policy_deductable,1000.0,1.136000e+03,6.118647e+02,500.00,500.0000,1000.0,2000.000,2000.00
policy_annual_premium,1000.0,1.256406e+03,2.441674e+02,433.33,1089.6075,1257.2,1415.695,2047.59
auto_year,1000.0,2.005103e+03,6.015861e+00,1995.00,2000.0000,2005.0,2010.000,2015.00


In [15]:
# Ratio de détection des valeurs extrêmes
num_cols["max_q75_ratio"] = num_cols["max"] / num_cols["75%"]

# Affichage des 10 premières variables potentielles aux outliers 
num_cols.sort_values("max_q75_ratio", ascending = False).head(15) # Ordre croissant

,count,mean,std,min,25%,50%,75%,max,max_q75_ratio
umbrella_limit,1000.0,1.101000e+06,2.297407e+06,-1000000.00,0.0000,0.0,0.000,10000000.00,inf
property_claim,1000.0,7.399570e+03,4.824726e+03,0.00,4445.0000,6750.0,10885.000,23670.00,2.174552
capital-gains,1000.0,2.512610e+04,2.787219e+04,0.00,0.0000,0.0,51025.000,100500.00,1.969623
injury_claim,1000.0,7.433420e+03,4.880952e+03,0.00,4295.0000,6775.0,11305.000,21450.00,1.897391
months_as_customer,1000.0,2.039540e+02,1.151132e+02,0.00,115.7500,199.5,276.250,479.00,1.733937
total_claim_amount,1000.0,5.276194e+04,2.640153e+04,100.00,41812.5000,58055.0,70592.500,114920.00,1.627935
vehicle_claim,1000.0,3.792895e+04,1.888625e+04,70.00,30292.5000,42100.0,50822.500,79560.00,1.565448
witnesses,1000.0,1.487000e+00,1.111335e+00,0.00,1.0000,1.0,2.000,3.00,1.500000
age,1000.0,3.894800e+01,9.140287e+00,19.00,32.0000,38.0,44.000,64.00,1.454545
policy_annual_premium,1000.0,1.256406e+03,2.441674e+02,433.33,1089.6075,1257.2,1415.695,2047.59,1.446350


>*Ces 10 premières variables méritent une attention très particulière en raison de leur valeurs très significatives, elles peuvent presenter des distributions assymétriques et influence énormement l'entrainement du modèle. Mais toutes ces assertions vont être vérifiées afin d'aller en profondeur et elles doivent faire l'`objet d'une analyse spécifique.`*

In [16]:
# Valeurs uniques de chaque colonne
data.nunique()

months_as_customer              391
age                              46
policy_number                  1000
policy_bind_date                951
policy_state                      3
policy_csl                        3
policy_deductable                 3
policy_annual_premium           991
umbrella_limit                   11
insured_zip                     995
insured_sex                       2
insured_education_level           7
insured_occupation               14
insured_hobbies                  20
insured_relationship              6
capital-gains                   338
capital-loss                    354
incident_date                    60
incident_type                     4
collision_type                    4
incident_severity                 4
authorities_contacted             4
incident_state                    7
incident_city                     7
incident_location              1000
incident_hour_of_the_day         24
number_of_vehicles_involved       4
property_damage             

>*De première vue, tous les lieux où les incidents se sont produits sont tous disctinctes.*

### 2-1. Que représente une ligne de chaque colonne ?
>Chaque ligne représente un sinistre d'assurance automobile avec les détails du client , de la police, de >l'incident et des montants réclamés. 

### 2-1-1. Dictionnaire de données

| Variable | Type | Description |
|----------|------|-------------|
| months_as_customer | int | Nombre de mois depuis lequel le client est assuré |
| age | int | Âge du client au moment du sinistre |
| policy_number | int | Numéro unique de la police d'assurance |
| policy_bind_date | datetime | Date de signature de la police |
| policy_state | object | État où la police a été émise |
| policy_csl | object | Couverture de responsabilité civile (Combined Single Limit) |
| policy_deductable | int | Montant de la franchise d'assurance |
| policy_annual_premium | float | Prime d'assurance annuelle en dollars |
| umbrella_limit | int | Limite de couverture supplémentaire (parapluie) en dollars |
| insured_zip | int | Code postal du client assuré |
| insured_sex | object | Sexe du client assuré (MALE/FEMALE) |
| insured_education_level | object | Niveau d'éducation du client (HS, MD, PhD) |
| insured_occupation | object | Profession du client assuré |
| insured_hobbies | object | Loisirs/hobbies du client |
| insured_relationship | object | Lien de parenté/relation du client |
| capital-gains | int | Gains en capital du client |
| capital-loss | int | Pertes en capital du client |
| incident_date | datetime | Date de l'incident/sinistre |
| incident_type | object | Type de sinistre (Single Vehicle Collision, Vehicle Theft, Multi-vehicle Collision) |
| collision_type | object | Type de collision (Front, Rear, Side Collision) |
| incident_severity | object | Gravité du sinistre (Minor, Major Damage, Total Loss) |
| authorities_contacted | object | Autorités contactées (Police, Tristate) |
| incident_state | object | État où s'est produit le sinistre |
| incident_city | object | Ville où s'est produit le sinistre |
| incident_location | object | Localisation spécifique du sinistre |
| incident_hour_of_the_day | int | Heure du jour du sinistre (0-23) |
| number_of_vehicles_involved | int | Nombre de véhicules impliqués dans le sinistre |
| property_damage | object | Présence de dommages matériels (YES/NO/?) |
| bodily_injuries | int | Nombre de blessures corporelles |
| witnesses | int | Nombre de témoins du sinistre |
| police_report_available | object | Disponibilité d'un rapport de police (YES/NO/?) |
| total_claim_amount | int | Montant total de la réclamation en dollars |
| injury_claim | int | Montant de la réclamation pour blessures corporelles |
| property_claim | int | Montant de la réclamation pour dommages matériels |
| vehicle_claim | int | Montant de la réclamation pour dommages véhicule |
| auto_make | object | Marque du véhicule impliqué |
| auto_model | object | Modèle du véhicule impliqué |
| auto_year | int | Année de fabrication du véhicule |
| fraud_reported | object | **[CIBLE]** Fraude déclarée (Y=Oui/N=Non) |

### 2-2. Qui a récolté ces données ?

>Ces données ont été récoltées par **Abdelrahim Aqqad** un auditeur interne en chef et un expert en comptabilité judiciaire basé à Doha, au Qatar, avec plus de 15 ans d'expérience dans l'audit opérationnel, financier et informatique, la gouvernance, la gestion des risques et les contrôles internes.
>
>Il est l'un des pionniers de l'utilisation de la *Data Science* et de l'*Analyse de Fraude* dans l'audit interne. 

### 2-3. Quand est-ce qu'elles ont été récoltées ?

> Ces données ont été récoltées en 2015

### 2-4. Où est-ce qu'elles ont été récoltées ?

> Ces données ont été récoltées en 2015 aux Etats Unis d'Amerique (USA)

### 2-5. Comment ont elles été récoltées ?

> Ces données ont été récoltées en 2015 aux Etats Unis d'Amerique (USA) via des compagnies de fournisseurs d'assurance.

### 2-6. Pourquoi est-ce qu'elles ont été récoltées ?

> Ces données ont été récoltées en 2015 aux Etats Unis d'Amerique (USA) via des compagnies de fournisseurs d'assurance dans le but de mener des travaux d'études dans le secteur de la finance pour des cas d'audit internes.